In [1]:
import numpy as np
from plasmapy.formulary import Debye_length
from astropy import units as u

# ── Physical Constants ─────────────────────────────────────
e      = 1.602e-19   # C
eps0   = 8.854e-12   # F/m
m_p    = 1.6726e-27  # kg
k_B    = 1.381e-23   # J/K

# ── Real Plasma Conditions ─────────────────────────────────
T_eV     = 1.0
T_K      = T_eV * 11604.5
n_e      = 1e19  # m⁻³

lambda_D = Debye_length(T_eV * u.eV, n_e * u.m**-3).to(u.m).value

# ── Normalize to Debye length & thermal energy ─────────────
# Length scale  : lambda_D
# Energy scale  : k_B * T
# These give dimensionless LJ-style units that LAMMPS handles well

kappa_norm = 1.0                              # κ * λ_D = 1 (by definition)
A_SI       = e**2 / (4 * np.pi * eps0)        # J·m
A_norm     = A_SI / (k_B * T_K * lambda_D)   # dimensionless Yukawa prefactor

# ── Simulation Box ─────────────────────────────────────────
N         = 100
box_size  = 10.0     # in units of λ_D
cutoff    = 3.0      # in units of λ_D
timestep  = 0.001    # in LJ time units (stable for Yukawa)
damping   = 0.1      # NVT damping
steps     = 5000
dump_freq = 25

print(f"Debye Length     : {lambda_D:.4e} m")
print(f"Temperature      : {T_K:.1f} K ({T_eV} eV)")
print(f"Yukawa A (norm)  : {A_norm:.4f}  (dimensionless)")
print(f"Screening κ (norm): {kappa_norm:.2f} (= 1/λ_D · λ_D)")
print(f"Frames to render : {steps // dump_freq}")

Debye Length     : 2.3508e-06 m
Temperature      : 11604.5 K (1.0 eV)
Yukawa A (norm)  : 0.0006  (dimensionless)
Screening κ (norm): 1.00 (= 1/λ_D · λ_D)
Frames to render : 200


In [2]:
from lammps import lammps

lmp = lammps()

lmp.commands_string(f"""
# ── Setup ──────────────────────────────────────────────────
units         lj
atom_style    atomic
boundary      p p p

# ── Box & Atoms ────────────────────────────────────────────
region        box block 0 {box_size} 0 {box_size} 0 {box_size}
create_box    1 box
create_atoms  1 random {N} 12345 box
mass          1 1.0

# ── Yukawa Potential (physically normalized) ───────────────
# κ = 1.0  →  screening length = 1 Debye length
# A = {A_norm:.4f} →  derived from e²/4πε₀ / (k_B·T·λ_D)
pair_style    yukawa {kappa_norm} {cutoff}
pair_coeff    1 1 {A_norm:.6f}

# ── Neighbor list ──────────────────────────────────────────
neigh_modify  one 5000 delay 0 every 1 check yes

# ── Initial Velocities ─────────────────────────────────────
velocity      all create 1.0 54321 dist gaussian

# ── NVT Thermostat ─────────────────────────────────────────
fix           1 all nvt temp 1.0 1.0 {damping}

# ── Dump Positions ─────────────────────────────────────────
dump          1 all custom {dump_freq} plasma.dump id x y z
dump_modify   1 sort id

# ── Run ────────────────────────────────────────────────────
timestep      {timestep}
thermo        {dump_freq}
run           {steps}
""")

lmp.close()
print("✓ LAMMPS simulation complete — plasma.dump written")

LAMMPS (29 Aug 2024)
OMP_NUM_THREADS environment is not set. Defaulting to 1 thread. (src/comm.cpp:98)
  using 1 OpenMP thread(s) per MPI task
Created orthogonal box = (0 0 0) to (10 10 10)
  1 by 1 by 1 MPI processor grid
Created 100 atoms
  using lattice units in orthogonal box = (0 0 0) to (10 10 10)
  create_atoms CPU = 0.000 seconds
Generated 0 of 0 mixed pair_coeff terms from geometric mixing rule
Neighbor list info ...
  update: every = 1 steps, delay = 0 steps, check = yes
  max neighbors/atom: 5000, page size: 100000
  master list distance cutoff = 3.3
  ghost atom cutoff = 3.3
  binsize = 1.65, bins = 7 7 7
  1 neighbor lists, perpetual/occasional/extra = 1 0 0
  (1) pair yukawa, perpetual
      attributes: half, newton on
      pair build: half/bin/atomonly/newton
      stencil: half/bin/3d
      bin: standard
Setting up Verlet run ...
  Unit style    : lj
  Current step  : 0
  Time step     : 0.001
Per MPI rank memory allocation (min/avg/max) = 3.066 | 3.066 | 3.066 Mbytes


In [3]:
def parse_lammps_dump(filepath):
    frames = []
    with open(filepath, "r") as f:
        lines = f.readlines()

    i = 0
    while i < len(lines):
        if "ITEM: TIMESTEP" in lines[i]:
            i += 2
            i += 2
            i += 4  # skip 3 box bound lines + header
            i += 1  # skip ITEM: ATOMS line

            positions = []
            while i < len(lines) and "ITEM:" not in lines[i]:
                parts = lines[i].split()
                x, y, z = float(parts[1]), float(parts[2]), float(parts[3])
                positions.append([x, y, z])
                i += 1
            frames.append(np.array(positions))
        else:
            i += 1
    return frames

frames = parse_lammps_dump("plasma.dump")

# Normalize positions to Debye lengths for readability
frames_normed = [f / lambda_D for f in frames]

print(f"✓ Parsed {len(frames)} frames, {len(frames[0])} protons each")
print(f"✓ Positions normalized to Debye lengths (λ_D = {lambda_D:.3e} m)")

✓ Parsed 201 frames, 100 protons each
✓ Positions normalized to Debye lengths (λ_D = 2.351e-06 m)


In [4]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation

box_norm = box_size / lambda_D  # normalized box size in λ_D units

fig = plt.figure(figsize=(7, 7), facecolor="#0a0a1a")
ax  = fig.add_subplot(111, projection='3d', facecolor="#0a0a1a")

def update(frame_idx):
    ax.cla()
    ax.set_facecolor("#0a0a1a")
    pos = frames_normed[frame_idx]

    ax.scatter(pos[:, 0], pos[:, 1], pos[:, 2],
               c='crimson', s=30, alpha=0.85,
               edgecolors='darkred', linewidths=0.3)

    ax.set_xlim(0, box_norm)
    ax.set_ylim(0, box_norm)
    ax.set_zlim(0, box_norm)
    ax.set_xlabel("x (λ_D)", color='white')
    ax.set_ylabel("y (λ_D)", color='white')
    ax.set_zlabel("z (λ_D)", color='white')
    ax.tick_params(colors='white')
    ax.set_title(
        f"Plasma Screening — 100 Protons\n"
        f"T = {T_eV} eV  |  n = {n_e:.0e} m⁻³  |  λ_D = {lambda_D:.2e} m\n"
        f"Frame {frame_idx + 1}/{len(frames_normed)}",
        color='white', fontsize=9
    )

ani = animation.FuncAnimation(fig, update, frames=len(frames_normed), interval=50)

writer = animation.FFMpegWriter(fps=20, bitrate=1800)
ani.save("plasma_screening.mp4", writer=writer)
plt.close()
print("✓ Saved plasma_screening.mp4")

✓ Saved plasma_screening.mp4
